<p><a href="https://query.farm/"><img src="https://raw.githubusercontent.com/Query-farm/vgi-example-notebooks/v0.1.1/assets/query-farm-strata-sun.svg" alt="Query.Farm — Strata Sun" width="252"></a></p>

<p><a href="https://query.farm/vgi/"><img src="https://raw.githubusercontent.com/Query-farm/vgi-example-notebooks/v0.1.1/assets/vgi-logo.png" alt="VGI" width="112"></a></p>

# [Professional Tennis Data](https://github.com/Query-farm/vgi-tennis)

ATP and WTA rankings, draws, Grand Slam finals and set-by-set results.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Query-farm/vgi-tennis/blob/main/notebooks/02_a_rank_cutoff_changes_the_points_pool.ipynb)

[Query.Farm](https://query.farm/) · [VGI](https://query.farm/vgi/) · [Example notes](https://github.com/Query-farm/vgi-tennis/blob/main/notebooks/README.md)

## A rank cutoff changes the points pool

A ranking position tells you an order, not the size of the points gap. Compare the mixed-tour convenience table with a bounded WTA request, then keep **one WTA singles ranking** fixed while selecting its top 20 or top 50. Count players and sum points separately.

ATP and WTA records come from different providers and can have different ranking dates. Their points are never pooled. This is a descriptive snapshot, not a win probability, ability estimate or prediction.

### Choose one change

Keep replay mode for a controlled comparison. Edit the parameter, restart and run all. Live source changes are a separate factor.

In [ ]:
DATA_MODE = "replay"  # Optional: "live", with no fallback
TOP_N = 20  # Experiment: 50

In [ ]:
import importlib.metadata as packages
import sys

installed = {d.metadata["Name"]: d.version for d in packages.distributions()}
if installed.get("vgi-example-notebooks") != "0.3.55":
    %pip install -q "git+https://github.com/Query-farm/vgi-example-notebooks@v0.3.55"
    if "vgi_notebooks" in sys.modules:
        raise RuntimeError("Restart the session after updating the example tools, then run all.")

In [ ]:
from vgi_notebooks import check_runtime, connect, finish, prepare, prose, theme

check_runtime()
prepare("tennis")
import hashlib
import json

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from tennis_example import TennisExample

theme()
session = connect(globals())
connection = session.connection

if type(TOP_N) is not int or not 1 <= TOP_N <= 100:
    raise ValueError("TOP_N must be an integer from 1 through 100")
example = TennisExample(DATA_MODE, "rankings")
chart_source = f"{DATA_MODE.title()} · complete public WTA/ESPN responses · v0.2.0\nCaptured independently; source times and population limits are shown in the tables."

### Connect the released worker

The short Git-tag command runs public **v0.2.0** in a separate environment. Shared setup preserves compatible notebook libraries and retrieves hash-verified example files. Replay serves only the recorded public response paths on loopback; unexpected requests fail. Live mode uses the actual public endpoints with no fallback. Endpoint overrides are restored at cleanup.

Data are provider-owned and independently acquired. The worker's code license does not license the source data. The calculations here describe bounded responses, not endorsed statistics or predictions.

In [ ]:
%%sql
INSTALL vgi FROM community;
LOAD vgi;
ATTACH 'tennis' (
  TYPE vgi,
  LOCATION 'uvx --python 3.13 --from git+https://github.com/Query-farm/vgi-tennis@v0.2.0 vgi-tennis'
);

### Retrieve once, then inspect source and date

`current_rankings` contains both tours. Materialize it before filtering so its true retrieved scope stays visible. The parameterized function requests WTA singles through rank 100. The worker caches identical underlying requests briefly; the replay audit verifies the actual three recorded HTTP responses. SQL row selection does not imply that only those rows were downloaded.

In [ ]:
%%sql
CREATE TABLE all_rankings AS SELECT * FROM tennis.main.current_rankings;
CREATE TABLE bounded AS SELECT * FROM tennis.main.rankings('WTA',ranking_type:='singles',max_rank:=100);
CREATE TABLE wta AS SELECT * FROM all_rankings WHERE tour='WTA' AND source='wta' AND ranking_type='singles';
CREATE TABLE selected AS SELECT * FROM bounded WHERE ranking<={{TOP_N}};

In [ ]:
%%sql scope <<
SELECT tour,source,ranking_type,min(ranked_at) AS first_date,max(ranked_at) AS last_date,
count(*) AS returned_players,min(ranking) AS first_rank,max(ranking) AS last_rank,count(points) AS known_points
FROM all_rankings GROUP BY tour,source,ranking_type ORDER BY tour;

In [ ]:
display(scope)
fingerprint = hashlib.sha256(json.dumps(connection.execute("SELECT * FROM wta ORDER BY ranking,player_id").fetchall(), default=str).encode()).hexdigest()
prose(f"The convenience table returned **{int(scope.returned_players.sum())} players** across two independently sourced tours. WTA has **{int(scope.loc[scope.tour.eq('WTA'), 'returned_players'].iloc[0])} rows**. The table shows each provider's ranking date; acquisition occurred from **{min(example.capture_times)}** through **{max(example.capture_times)}** in replay mode. WTA fingerprint: `{fingerprint}`. Live mode records a new run time of {example.started_at}; archived acquisition times do not describe a live response.")

### Check identity before comparing the two access paths

Join on tour, source, ranking type and player ID. Names are labels, and IDs from ESPN and WTA are different namespaces. Verify that the first 100 WTA rows agree on rank, points and ranking date. This agreement is about these materialized responses; separately fetched live snapshots can change between requests.

In [ ]:
%%sql agreement <<
SELECT count(*) AS requested_rows,count(w.player_id) AS matched_rows,
count(*) FILTER(WHERE b.ranking IS DISTINCT FROM w.ranking OR b.points IS DISTINCT FROM w.points OR b.ranked_at IS DISTINCT FROM w.ranked_at) AS mismatches
FROM bounded b LEFT JOIN wta w USING(tour,source,ranking_type,player_id);

In [ ]:
display(agreement)
assert agreement.requested_rows.iloc[0] == agreement.matched_rows.iloc[0] == 100
assert agreement.mismatches.iloc[0] == 0
prose("All **100 requested WTA players** match the convenience table on source-qualified identity, rank, points and ranking date. The next selection therefore changes the comparison population, not the source values.")

### Separate player share from point share

The denominator is the complete 200-row WTA ranking returned by the convenience table. Sum only known points and show how many values are known; a missing value is never replaced by zero. The selected top-N population is a nested subset. Its share of ranking points is a descriptive concentration measure, not its share of future wins.

In [ ]:
%%sql summary <<
SELECT (SELECT count(*) FROM wta) AS full_players,(SELECT count(points) FROM wta) AS full_known,
(SELECT sum(points) FROM wta) AS full_points,count(*) AS selected_players,count(points) AS selected_known,
sum(points) AS selected_points,100.0*count(*)/(SELECT count(*) FROM wta) AS player_share_pct,
100.0*sum(points)/(SELECT sum(points) FROM wta) AS point_share_pct FROM selected;

In [ ]:
display(summary)
r = summary.iloc[0]
prose(f"The top **{int(r.selected_players)}** players are **{r.player_share_pct:.1f}%** of this {int(r.full_players)}-row WTA population but hold **{r.point_share_pct:.1f}%** of its returned points. All **{int(r.full_known)}** point values are known. Expanding the cutoff changes both numerators while the full-ranking denominator remains fixed.")

In [ ]:
%%sql curve <<
SELECT ranking,player_name,points,ranking<={{TOP_N}} AS selected,
sum(points) OVER(ORDER BY ranking,player_id ROWS UNBOUNDED PRECEDING) AS cumulative_points,
100.0*sum(points) OVER(ORDER BY ranking,player_id ROWS UNBOUNDED PRECEDING)/sum(points) OVER() AS point_share_pct,
100.0*row_number() OVER(ORDER BY ranking,player_id)/count(*) OVER() AS player_share_pct
FROM wta ORDER BY ranking,player_id;

In [ ]:
from matplotlib.ticker import StrMethodFormatter

part = curve[curve.ranking.le(50)]
fig, ax = plt.subplots(figsize=(11.8, 7))
ax.plot(part.ranking, part.points, color="#A5BFB6", linewidth=1.4)
ax.scatter(part.ranking, part.points, c=np.where(part.selected, "#217C67", "#A5BFB6"), s=36, zorder=3)
ax.set(xlabel="Published WTA rank (top 50 shown)", ylabel="Ranking points (count)", xlim=(0,52), ylim=(0,float(part.points.max())*1.13))
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
finish(fig, "Adjacent ranks need not have similar point totals", f"WTA singles · selected top {TOP_N} in green · fixed top-50 display", chart_source, name=f"ranking-points-{TOP_N}")

In [ ]:
fig, ax = plt.subplots(figsize=(11.8, 7))
ax.plot(np.r_[0,curve.player_share_pct], np.r_[0,curve.point_share_pct], color="#217C67", linewidth=2)
ax.plot([0,100],[0,100], linestyle="--", color="#A5BFB6", label="Equal points per returned player")
ax.scatter([r.player_share_pct],[r.point_share_pct], color="#C5843E", s=75, zorder=3)
ax.annotate(f"Top {TOP_N}: {r.player_share_pct:.1f}% of players\n{r.point_share_pct:.1f}% of returned points",(r.player_share_pct,r.point_share_pct),xytext=(47,34),arrowprops={"arrowstyle":"-","color":"#C5843E"},bbox={"facecolor":"white","edgecolor":"none","pad":3})
ax.set(xlabel="Share of returned WTA players, highest rank first (%)", ylabel="Share of returned WTA ranking points (%)", xlim=(0,100), ylim=(0,104))
ax.legend(loc="lower right", fontsize=9)
finish(fig, "The cutoff selects a different share of the points pool", "Fixed 200-row denominator · ranking points are not probabilities", chart_source, name=f"ranking-share-{TOP_N}")

In [ ]:
%%sql checks <<
SELECT 'Source-qualified current keys unique' AS check_name,count(*)=count(DISTINCT(tour,source,ranking_type,player_id)) AS passed FROM all_rankings
UNION ALL SELECT 'WTA population complete',count(*)=200 AND count(points)=200 AND count(DISTINCT ranking)=200 FROM wta
UNION ALL SELECT 'WTA ranking date known and common',count(ranked_at)=200 AND count(DISTINCT ranked_at)=1 FROM wta
UNION ALL SELECT 'Known nonnegative points',count(points)=count(*) AND bool_and(points>=0) FROM wta
UNION ALL SELECT 'Bounded rows present',count(*)=100 AND min(ranking)=1 AND max(ranking)=100 FROM bounded
UNION ALL SELECT 'Nested selected roster',count(*)=0 FROM selected s ANTI JOIN wta w USING(tour,source,ranking_type,player_id)
UNION ALL SELECT 'Requested cutoff applied',count(*)={{TOP_N}} AND max(ranking)={{TOP_N}} FROM selected
UNION ALL SELECT 'Point sum positive',sum(points)>0 FROM wta
UNION ALL SELECT 'Ranking points nonincreasing',count(*)=0 FROM (SELECT points,lag(points) OVER(ORDER BY ranking) AS previous FROM wta) WHERE points>previous;

In [ ]:
display(checks)
assert checks.passed.notna().all() and checks.passed.all()
assert np.isclose(curve.point_share_pct.iloc[-1],100) and curve.point_share_pct.is_monotonic_increasing
prose(f"All **{len(checks)} SQL checks** and cumulative-share endpoints pass. " + example.audit())

### Experiment: expand the selected ranking

Change only `TOP_N=20` to **50**. Predict the player share exactly and the direction of the point-share change. Compare the green markers and highlighted cumulative point. The 200 source rows, ranking dates, source fingerprint, points and cumulative curve must stay unchanged; only the selected marker and population change. Restore 20 afterward.

In [ ]:
session.close()
example.close()

<a href="https://query.farm/"><img src="https://raw.githubusercontent.com/Query-farm/vgi-example-notebooks/v0.1.1/assets/query-farm-strata-sun.svg" alt="Query.Farm — Strata Sun" width="252"></a>

[Query.Farm](https://query.farm/) · [VGI](https://query.farm/vgi/) · [Worker repository: vgi-tennis](https://github.com/Query-farm/vgi-tennis)